# Step 7: Feature Engineering

●	Total order value

●	Delivery days

●	Delivery delay

●	Customer order count

●	Customer total spending

●	Average order value

●	Seller revenue

●	Seller order count

●	Repeat customer indicator



In this step, new business features are created from the cleaned datasets to support further analysis and business insights.

In [1]:
import pandas as pd

orders = pd.read_csv("../Data/2_cleaned/orders_clean.csv")
order_items = pd.read_csv("../Data/2_cleaned/order_items_clean.csv")
payments = pd.read_csv("../Data/2_cleaned/payments_clean.csv")
customers = pd.read_csv("../Data/2_cleaned/customers_clean.csv")
sellers = pd.read_csv("../Data/2_cleaned/sellers_clean.csv")

## 1. Total Order Value

Calculate the total value of each order using product price and freight value.

In [2]:
order_items["item_total"] = (
    order_items["price"] + order_items["freight_value"]
)

calculate the total for each order_id

In [3]:
order_value = (
    order_items
    .groupby("order_id")["item_total"]
    .sum()
    .reset_index()
)

order_value.rename(
    columns={"item_total": "total_order_value"},
    inplace=True
)

order_value.head()

,order_id,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


# 2. Delivery Days.



Calculate the number of days between the order purchase date and the actual delivery date.

In [4]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)

create the feature:

In [5]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [6]:
orders[
    [
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_days"
    ]
].head()

,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
1,2018-07-24 20:41:37,2018-08-07 15:27:45,13.0
2,2018-08-08 08:38:49,2018-08-17 18:06:29,9.0
3,2017-11-18 19:28:06,2017-12-02 00:28:42,13.0
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2.0


# 3. Delivery Delay.


Calculate how many days an order was delivered after the estimated delivery date.

In [ ]:
# convert the estimated delivery date:
orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
)

In [8]:
# create the feature:
orders["delivery_delay"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

In [9]:
# Check
orders[
    [
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_delay"
    ]
].head()

,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay
0,2017-10-10 21:25:13,2017-10-18,-8.0
1,2018-08-07 15:27:45,2018-08-13,-6.0
2,2018-08-17 18:06:29,2018-09-04,-18.0
3,2017-12-02 00:28:42,2017-12-15,-13.0
4,2018-02-16 18:17:02,2018-02-26,-10.0


# 4. Customer Order Count


Calculate how many orders each customer has placed.

In [14]:
orders_customer = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

In [15]:
customer_order_count = (
    orders_customer
    .groupby("customer_unique_id")["order_id"]
    .nunique()
    .reset_index()
)

customer_order_count.rename(
    columns={"order_id": "customer_order_count"},
    inplace=True
)

customer_order_count.head()

,customer_unique_id,customer_order_count
0,0000366f3b9a7992bf8c76cfdf3221e2,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1
2,0000f46a3911fa3c0805444483337064,1
3,0000f6ccb0745a6a4b88665a16c9f078,1
4,0004aac84e0df4da2b147fca70cf8255,1


# 5. Customer Total Spending.

Calculate the total amount spent by each unique customer.

In [16]:
customer_spending = orders_customer.merge(
    payments[["order_id", "payment_value"]],
    on="order_id",
    how="left"
)

In [17]:
customer_total_spending = (
    customer_spending
    .groupby("customer_unique_id")["payment_value"]
    .sum()
    .reset_index()
)

customer_total_spending.rename(
    columns={"payment_value": "customer_total_spending"},
    inplace=True
)

customer_total_spending.head()

,customer_unique_id,customer_total_spending
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19
2,0000f46a3911fa3c0805444483337064,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,43.62
4,0004aac84e0df4da2b147fca70cf8255,196.89


# 6. Average Order Value.

 average, how much money does a customer spend per order?
 
 if a customer spent ₹600 across 3 orders:

Calculate the average amount spent per order by each customer.

# Combine the two features you already created

In [22]:
customer_features = customer_total_spending.merge(
    customer_order_count,
    on="customer_unique_id",
    how="left"
)

# Create Average Order Value

In [23]:
customer_features["average_order_value"] = (
    customer_features["customer_total_spending"]
    / customer_features["customer_order_count"]
)

In [24]:
customer_features[
    [
        "customer_unique_id",
        "customer_total_spending",
        "customer_order_count",
        "average_order_value"
    ]
].head()

,customer_unique_id,customer_total_spending,customer_order_count,average_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19,1,27.19
2,0000f46a3911fa3c0805444483337064,86.22,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,43.62,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,196.89,1,196.89


# 7. Seller Revenue

How much product sales value each seller generated.

Calculate the total sales revenue generated by each seller.

# Calculate Seller Revenue

In [26]:
seller_revenue = (
    order_items
    .groupby("seller_id")["price"]
    .sum()
    .reset_index()
)

In [27]:
seller_revenue.rename(
    columns={"price": "seller_revenue"},
    inplace=True
)

# Check the result

In [28]:
seller_revenue.head()

,seller_id,seller_revenue
0,0015a82c2db000af6aaaf3ae2ecb0532,2685.00
1,001cca7ae9ae17fb1caed9dfb1094831,25080.03
2,001e6ad469a905060d959994f1b41e4f,250.00
3,002100f778ceb8431b7a1020ff7ab48f,1234.50
4,003554e2dce176b5555353e4f3555ac8,120.00


# 8. Seller Order Count.

Calculate how many unique orders each seller handled.

In [29]:
seller_order_count = (
    order_items
    .groupby("seller_id")["order_id"]
    .nunique()
    .reset_index()
)

In [30]:
# Rename the column:
seller_order_count.rename(
    columns={"order_id": "seller_order_count"},
    inplace=True
)

In [31]:
seller_order_count.head()

,seller_id,seller_order_count
0,0015a82c2db000af6aaaf3ae2ecb0532,3
1,001cca7ae9ae17fb1caed9dfb1094831,200
2,001e6ad469a905060d959994f1b41e4f,1
3,002100f778ceb8431b7a1020ff7ab48f,51
4,003554e2dce176b5555353e4f3555ac8,1


# 9. Repeat Customer Indicator.

Identify whether a customer has placed more than one order.


In [32]:
# already created customer_order_count, use it to make the new feature:
customer_order_count["repeat_customer"] = (
    customer_order_count["customer_order_count"] > 1
)

In [33]:
customer_order_count.head()

,customer_unique_id,customer_order_count,repeat_customer
0,0000366f3b9a7992bf8c76cfdf3221e2,1,False
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,False
2,0000f46a3911fa3c0805444483337064,1,False
3,0000f6ccb0745a6a4b88665a16c9f078,1,False
4,0004aac84e0df4da2b147fca70cf8255,1,False


In [ ]:
# i  prefer clearer labels instead of True/False
customer_order_count["repeat_customer"] = (
    customer_order_count["customer_order_count"]
    .apply(lambda x: "Yes" if x > 1 else "No")
)

## Final Feature Tables

In [35]:
# combine the order features

orders_features = orders.merge(
    order_value,
    on="order_id",
    how="left"
)

In [36]:
# nother Code cell for customer features:
customer_features = customer_features.merge(
    customer_order_count[
        ["customer_unique_id", "repeat_customer"]
    ],
    on="customer_unique_id",
    how="left"
)

# seller features:

In [39]:
seller_features = seller_revenue.merge(
    seller_order_count,
    on="seller_id",
    how="left"
)

# Finally check all three:-

- orders_features
- customer_features
- seller_features

In [40]:
orders_features.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay,total_order_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,28.62


In [41]:
customer_features.head()

,customer_unique_id,customer_total_spending,customer_order_count,average_order_value,repeat_customer
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90,1,141.90,No
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19,1,27.19,No
2,0000f46a3911fa3c0805444483337064,86.22,1,86.22,No
3,0000f6ccb0745a6a4b88665a16c9f078,43.62,1,43.62,No
4,0004aac84e0df4da2b147fca70cf8255,196.89,1,196.89,No


In [42]:
seller_features.head()

,seller_id,seller_revenue,seller_order_count
0,0015a82c2db000af6aaaf3ae2ecb0532,2685.00,3
1,001cca7ae9ae17fb1caed9dfb1094831,25080.03,200
2,001e6ad469a905060d959994f1b41e4f,250.00,1
3,002100f778ceb8431b7a1020ff7ab48f,1234.50,51
4,003554e2dce176b5555353e4f3555ac8,120.00,1
